# Used Bike Price Prediction Project

## Problem Statement
The growth of the pre-owned two-wheeler market has increased the need for fair, data-driven price valuation. Using the provided dataset, perform:

1. Data Cleaning and Exploratory Data Analysis (EDA)
2. Feature Engineering
3. Regression Model Development
4. Model Comparison using MAE, RMSE, and R²
5. Feature Importance Analysis
6. Business Recommendations

### Research Questions
- What factors most strongly influence resale price?
- How does depreciation vary by brand and engine power?
- Do urban markets show price premiums?
- How does original price relate to resale price?
- Are there pricing differences across ownership types?


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv('used bike price data set.csv')

print('Shape:', df.shape)
display(df.head())
display(df.info())
display(df.describe(include='all'))


## Data Cleaning & Feature Engineering

In [ ]:
# Missing values
print(df.isnull().sum())

# Remove duplicates
df = df.drop_duplicates()

# Feature Engineering
df['depreciation'] = df['Original Price'] - df['price']
df['retention_ratio'] = df['price'] / df['Original Price']
df['kms_per_year'] = df['kms_driven'] / (df['age'] + 1)

df.head()


## Exploratory Data Analysis

In [ ]:
# Correlation Analysis
numeric_cols = ['price','kms_driven','age','power','Original Price',
                'depreciation','retention_ratio','kms_per_year']

corr = df[numeric_cols].corr()

plt.figure(figsize=(10,6))
sns.heatmap(corr, annot=True, cmap='coolwarm')
plt.title('Correlation Matrix')
plt.show()


In [ ]:
# Brand-wise average price
brand_price = df.groupby('brand')['price'].mean().sort_values(ascending=False)

plt.figure(figsize=(12,5))
brand_price.head(15).plot(kind='bar')
plt.title('Average Resale Price by Brand')
plt.ylabel('Price')
plt.show()


In [ ]:
# Ownership impact
plt.figure(figsize=(8,5))
sns.boxplot(data=df, x='owner', y='price')
plt.xticks(rotation=20)
plt.title('Price Distribution by Ownership Type')
plt.show()


In [ ]:
# City-wise premium
city_avg = df.groupby('city')['price'].mean().sort_values(ascending=False)

plt.figure(figsize=(12,5))
city_avg.head(15).plot(kind='bar')
plt.title('Average Bike Price by City')
plt.show()


## Machine Learning Models

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor

X = df.drop('price', axis=1)
y = df['price']

categorical = ['bike_name','city','owner','brand']
numeric = [col for col in X.columns if col not in categorical]

preprocessor = ColumnTransformer([
    ('cat', OneHotEncoder(handle_unknown='ignore'), categorical),
    ('num', 'passthrough', numeric)
])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

models = {
    'Linear Regression': LinearRegression(),
    'Random Forest': RandomForestRegressor(n_estimators=200, random_state=42),
    'Gradient Boosting': GradientBoostingRegressor(random_state=42)
}

results = []

for name, model in models.items():
    pipe = Pipeline([
        ('prep', preprocessor),
        ('model', model)
    ])

    pipe.fit(X_train, y_train)
    pred = pipe.predict(X_test)

    mae = mean_absolute_error(y_test, pred)
    rmse = np.sqrt(mean_squared_error(y_test, pred))
    r2 = r2_score(y_test, pred)

    results.append([name, mae, rmse, r2])

results_df = pd.DataFrame(
    results,
    columns=['Model','MAE','RMSE','R2']
).sort_values('R2', ascending=False)

results_df


## Best Model & Feature Importance

In [ ]:
best_model = Pipeline([
    ('prep', preprocessor),
    ('model', RandomForestRegressor(
        n_estimators=200,
        random_state=42
    ))
])

best_model.fit(X_train, y_train)

rf = best_model.named_steps['model']

feature_names = best_model.named_steps['prep'].get_feature_names_out()

importance = pd.DataFrame({
    'Feature': feature_names,
    'Importance': rf.feature_importances_
}).sort_values('Importance', ascending=False)

importance.head(20)


## Expected Conclusions

1. Original Price is usually the strongest predictor of resale value.
2. Age negatively impacts selling price due to depreciation.
3. Higher engine power generally increases resale value.
4. Kilometers driven has a negative effect on price.
5. Premium brands retain value better than economy brands.
6. Major cities may show resale price premiums.
7. First-owner bikes generally command higher prices.

## Evaluation Metrics

- MAE (Mean Absolute Error)
- RMSE (Root Mean Squared Error)
- R² Score

The model with the highest R² and lowest MAE/RMSE should be selected for deployment.
